# Building a Model-as-Judge Evaluation Pipeline for Production LLM Systems

You changed a prompt to fix one annoying edge case. It looks better on the three examples you tried. Do you ship it?

Without an evaluation pipeline, that question has no good answer. "It looks right" scales to about five examples and one reviewer; a production LLM feature serves thousands of inputs you will never eyeball. The gap between those two numbers is where regressions live — the reworded prompt that fixed your edge case and quietly broke numerical reasoning, the model upgrade that improved fluency and started fabricating figures, the "small" system-prompt tweak that made the assistant answer questions it should have declined.

This cookbook builds the machinery that answers "do you ship it?" with a number and a threshold instead of a vibe. It is **not** a benchmark reproduction — it is the eval harness a team puts around a real feature: a held-out golden dataset, a model-as-judge scorer that decomposes quality into independent dimensions, a regression gate that blocks a bad change in CI, and a shadow-evaluation pattern for comparing a candidate against production before promoting it.

The worked domain is financial-document Q&A — extracting facts, running calculations, interpreting compliance rules, and (crucially) declining to answer when a document does not contain the answer. It is the same regulated setting as the compliance-aware agent cookbook that accompanies this one in the series: that one builds a careful agent, this one shows how you would prove it stays careful across every future change.

## By the end of this cookbook, you'll be able to:

- **Design** a held-out golden dataset with rubrics and metadata, and understand why "held-out" is load-bearing.
- **Score** open-ended answers with a model-as-judge that breaks quality into independent dimensions — correctness, completeness, hallucination, tone — one judge call per dimension to avoid the halo effect.
- **Gate** a prompt or model change behind hard thresholds so a regression cannot merge.
- **Compare** a candidate model against production with a shadow-evaluation A/B, and know when to promote.
- **Wire** the whole thing into CI/CD as an exit-0/exit-1 script that runs on the pull requests that touch your prompts.

## Prerequisites

**Required knowledge**
- Comfortable with the Messages API (system prompts, `messages.create`).
- Basic familiarity with pull-request CI (something runs on your PR and can block the merge).

**Required setup**
- An Anthropic API key in the environment as `ANTHROPIC_API_KEY`.
- The two helper modules that ship next to this notebook: `utils/evaluator.py` (the scoring contract) and `prompts.py` (the production prompt under test).

The judge and the system under test both run on `claude-haiku-4-5` — evaluation is high-volume, and a cheap, fast judge is what makes it affordable to run on every PR.

## Setup

In [1]:
%%capture
%pip install -U "anthropic>=0.109.0" pandas numpy python-dotenv

In [2]:
import json
import os
import subprocess
import sys
from pathlib import Path

import anthropic
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from utils.evaluator import (
    DEFAULT_THRESHOLDS,
    DIMENSIONS,
    DOCUMENTS,
    GOLDEN_DATASET,
    JUDGE_PROMPT_TEMPLATE,
    aggregate,
    check_regression_gate,
    evaluate_response,
    run_suite,
)

load_dotenv()

# The system under test and the judge both run on cheap, fast Haiku — evaluation
# is high-volume, so an expensive judge would make running eval on every PR
# unaffordable. We hold the judge model FIXED so scores stay comparable as the
# thing being evaluated changes.
SUT_MODEL = "claude-haiku-4-5"
JUDGE_MODEL = "claude-haiku-4-5"

client = anthropic.Anthropic(max_retries=16)

DATA_DIR = Path.cwd()

## 1. Why "it looks right" isn't a release criterion

Manual spot-checking feels like evaluation, but it fails in three specific ways the moment a feature reaches production scale.

**It doesn't scale, and it isn't reproducible.** A reviewer can eyeball five outputs. They cannot eyeball five hundred, and two reviewers will disagree on the borderline ones. Worse, the review isn't a record: when someone asks in three months "did the March prompt change hurt compliance answers?", there is nothing to re-run. Spot-checking produces a feeling, not a measurement.

**It only inspects the outputs you happen to look at.** People spot-check the happy path — the inputs they thought of. The failures that reach users are the ones nobody thought to try: the malformed document, the question phrased three unusual ways, the edge case that only shows up on 2% of traffic.

**A golden dataset turns evaluation into a measurement you can repeat.** A *golden dataset* is a fixed, curated set of inputs paired with known-good expected outputs and a rubric for what "good" means. Its most important property is that it is **held out**: it is not used to write or tune the prompt. If you tune your prompt against the same examples you score it on, a high score means the prompt memorized the test — the number goes up while real quality does not. A held-out set is the only kind whose score you can trust to move with reality.

With a golden dataset in hand, evaluation catches the three failure modes that would otherwise be caught by *users*:

| Failure mode | What it looks like | How eval catches it |
| --- | --- | --- |
| **Regression** | A change that improves one thing silently breaks another. | Scores on the unchanged categories drop. |
| **Edge-case drift** | The rare input the happy-path review never covered. | The golden set deliberately includes edge cases (here: unanswerable questions). |
| **Prompt-change side effects** | A reworded instruction shifts behavior far from where you were looking. | Every dimension is re-scored, so a tone tweak that hurt correctness shows up. |

The rest of this notebook builds each piece: the dataset (§2), the judge (§3), the suite (§4), the gate (§5), shadow evaluation (§6), and the CI wiring (§7).

## 2. Building a golden dataset

A golden example is more than an input–output pair. Each record carries five things:

- **`input`** — here, a source document plus a question about it.
- **`expected`** — a reference answer a domain expert would accept.
- **`rubric`** — a short, explicit statement of what a correct answer *must* contain. This is what the judge grades against, and being explicit here is what makes scoring consistent.
- **`metadata`** — `category`, `difficulty`, and an `edge_case` flag, so you can slice scores by where quality matters and track whether the hard cases are covered.

Our dataset has 12 questions over three short financial documents, evenly spread across four categories (a teaching size — a production golden set is hundreds of examples, and grows every time a new failure reaches users):

- **factual_extraction** — pull a stated value from the document.
- **numerical_reasoning** — compute something (an LTV ratio, gross proceeds in the right units).
- **compliance_interpretation** — apply a regulatory rule to the document.
- **ambiguous_unanswerable** — the document does *not* contain the answer; the only correct move is to decline. These are the hallucination tripwires, and they are the examples a happy-path review never writes.

The dataset lives in `utils/evaluator.py` so it can be imported by both this notebook and the CI script. Let's look at one record from each category:

In [3]:
for category in [
    "factual_extraction",
    "numerical_reasoning",
    "compliance_interpretation",
    "ambiguous_unanswerable",
]:
    record = next(r for r in GOLDEN_DATASET if r["category"] == category)
    print(
        f"=== {category} ({record['id']}, difficulty={record['difficulty']}, edge_case={record['edge_case']}) ==="
    )
    print(f"Q: {record['question']}")
    print(f"Expected: {record['expected']}")
    print(f"Rubric:   {record['rubric']}\n")

=== factual_extraction (fe-1, difficulty=easy, edge_case=False) ===
Q: What is the applicant's stated gross annual income?
Expected: GBP 72,000 per year.
Rubric:   Must state 72,000 GBP (currency and amount). Extra context is fine.

=== numerical_reasoning (nr-1, difficulty=medium, edge_case=False) ===
Q: What is the loan-to-value (LTV) ratio for this mortgage?
Expected: 75% (a 300,000 loan against a 400,000 property).
Rubric:   Must arrive at 75%. Accept the correct ratio even if the working is brief.

=== compliance_interpretation (ci-1, difficulty=medium, edge_case=False) ===
Q: Under MiFID II suitability rules, can the firm make a personal recommendation if it cannot obtain the client's financial situation and objectives?
Expected: No. If the firm cannot obtain the required information it must not make a personal recommendation.
Rubric:   Must conclude the firm must NOT proceed with a personal recommendation. A 'yes' or hedged answer is wrong.

=== ambiguous_unanswerable (au-1, dif

Before trusting a golden set, look at its shape. A dataset that is all easy factual questions will give you a reassuring score that means nothing. We want real coverage: an even spread across categories, a mix of difficulties, and a meaningful fraction of edge cases.

In [4]:
df_golden = pd.DataFrame(GOLDEN_DATASET)

print("By category:")
print(df_golden["category"].value_counts().to_string())
print("\nBy difficulty:")
print(df_golden["difficulty"].value_counts().to_string())
edge_pct = 100 * df_golden["edge_case"].mean()
print(f"\nEdge cases: {df_golden['edge_case'].sum()} / {len(df_golden)} ({edge_pct:.0f}%)")

By category:
category
factual_extraction           3
numerical_reasoning          3
compliance_interpretation    3
ambiguous_unanswerable       3

By difficulty:
difficulty
medium    6
hard      4
easy      2

Edge cases: 3 / 12 (25%)


Finally, persist the dataset to JSON with a schema check. Saving it as a versioned artifact is what makes the eval reproducible — the CI script loads this exact file — and the validation guards against a malformed record silently corrupting a run.

In [5]:
REQUIRED_FIELDS = {
    "id",
    "document",
    "question",
    "expected",
    "rubric",
    "category",
    "difficulty",
    "edge_case",
}


def validate_golden(dataset: list[dict]) -> None:
    """Fail loudly if any record is malformed — a bad golden set poisons every score."""
    seen_ids = set()
    for record in dataset:
        missing = REQUIRED_FIELDS - record.keys()
        if missing:
            raise ValueError(f"record {record.get('id', '?')} missing fields: {missing}")
        if record["id"] in seen_ids:
            raise ValueError(f"duplicate id: {record['id']}")
        if record["document"] not in DOCUMENTS:
            raise ValueError(
                f"record {record['id']} references unknown document {record['document']!r}"
            )
        seen_ids.add(record["id"])


validate_golden(GOLDEN_DATASET)
golden_path = DATA_DIR / "golden_dataset.json"
golden_path.write_text(json.dumps(GOLDEN_DATASET, indent=2))
print(f"Validated and saved {len(GOLDEN_DATASET)} golden examples -> {golden_path.name}")

Validated and saved 12 golden examples -> golden_dataset.json


## 3. The model-as-judge pattern

### Why F1 isn't enough

For a classification task you can compute exact-match accuracy or F1 against the label. Financial-document answers are open-ended: "GBP 72,000 a year", "£72k per annum", and "The stated gross income is 72,000 pounds" are all correct, and string metrics reject two of them. You could hand-write matchers, but they break the moment an answer is phrased a new way, and they cannot judge the things that actually matter here — did the answer *hallucinate*, is the *tone* appropriate for a regulated setting.

A **model-as-judge** reads the question, the reference answer, the rubric, and the candidate answer, and scores the candidate the way a human grader would. It generalizes across phrasing and can assess qualities no regex can.

### Decompose quality into independent dimensions

The naive judge asks for one overall score. That invites the **halo effect**: a fluent, confident-sounding answer gets marked correct because it *reads* well, even when its numbers are wrong. We avoid it by scoring four independent dimensions, each in its **own judge call** so no dimension's judgement can bleed into another:

- **correctness** — is it factually right versus the reference and document?
- **completeness** — does it cover everything the question and rubric ask for?
- **hallucination** — is every claim grounded in the document? (5 = fully grounded, *including correctly declining* an unanswerable question; 1 = fabricated.)
- **tone** — is the register right for a regulated financial setting?

Every dimension is scored 1–5 with higher always better, and each has an explicit scale so the judge is consistent. Here they are, with the weights used to roll them up into an overall score:

In [6]:
df_dims = pd.DataFrame(
    [
        {"dimension": d.name, "weight": d.weight, "what it measures": d.description[:70] + "..."}
        for d in DIMENSIONS
    ]
)
print(df_dims.to_string(index=False))

    dimension  weight                                                          what it measures
  correctness    0.40 Is the answer factually right relative to the reference answer and the...
 completeness    0.25      Does the answer cover everything the question and rubric ask for?...
hallucination    0.25 Is every claim grounded in the source document? Penalize invented figu...
         tone    0.10 Is the register appropriate for a regulated financial setting: precise...


### The judge prompt

The prompt for a single dimension does four things that make its scores trustworthy: it isolates **one** dimension, gives an explicit **1–5 scale**, requires **chain-of-thought before the score** (so the reasoning informs the number rather than rationalizing it after), and demands a rigid **XML format** we can parse without ambiguity.

In [7]:
print(JUDGE_PROMPT_TEMPLATE[:1100] + "\n...")

You are a meticulous evaluator scoring ONE quality dimension of an answer to a question about a financial document. Score only the dimension named below; ignore every other quality.

Dimension: {dimension_name}
{dimension_description}

Scoring scale (integer 1-5):
{dimension_scale}

<source_document>
{document}
</source_document>

<question>
{question}
</question>

<reference_answer>
{expected}
</reference_answer>

<rubric>
{rubric}
</rubric>

<candidate_answer>
{actual}
</candidate_answer>

First reason step by step about how the candidate answer measures up on the {dimension_name} dimension only. Then output an integer score from 1 to 5.

Reply in EXACTLY this XML format and nothing else:
<evaluation>
<reasoning>two to four sentences of assessment</reasoning>
<score>an integer from 1 to 5</score>
</evaluation>
...


`evaluate_response` runs one judge call per dimension and returns an `EvalResult` with the four scores, the judge's reasoning for each, and a weighted overall. Let's score one strong answer to the LTV question:

In [8]:
ltv_q = next(r for r in GOLDEN_DATASET if r["id"] == "nr-1")
good_answer = (
    "The loan-to-value ratio is 75%. The requested loan is GBP 300,000 against a "
    "property valued at GBP 400,000, and 300,000 / 400,000 = 0.75."
)

result = evaluate_response(
    client,
    JUDGE_MODEL,
    question_id=ltv_q["id"],
    category=ltv_q["category"],
    document=DOCUMENTS[ltv_q["document"]],
    question=ltv_q["question"],
    expected=ltv_q["expected"],
    rubric=ltv_q["rubric"],
    actual=good_answer,
)

print(f"scores : {result.scores}")
print(f"overall: {result.overall:.2f}\n")
for dim, reasoning in result.rationales.items():
    print(f"[{dim}] {reasoning}")

scores : {'correctness': 5, 'completeness': 5, 'hallucination': 5, 'tone': 5}
overall: 5.00

[correctness] The candidate answer correctly identifies the LTV ratio as 75%, which matches the reference answer exactly. The working is shown clearly: the loan amount (GBP 300,000) is divided by the property value (GBP 400,000) to arrive at 0.75 or 75%. This calculation is mathematically accurate and properly sourced from the document figures. There are no material errors or omissions.
[completeness] The candidate answer addresses the question by calculating and stating the correct LTV ratio of 75%. The rubric requires arriving at 75%, which the candidate does. The answer also includes the supporting working (300,000 / 400,000 = 0.75) and identifies the relevant figures from the source document. All required elements specified in the rubric are present: the correct ratio and the calculation basis.
[hallucination] The candidate answer states that the loan-to-value ratio is 75%, calculated as GB

### Does the judge agree with humans?

A judge is only useful if its scores track what a human grader would say. You validate that once, up front, on a small hand-labelled set: score each answer with the judge, and correlate against the human label. Here are six answers of deliberately varying quality — from a perfect extraction to a confidently fabricated credit score — each with a simulated human overall rating (1–5):

In [9]:
# (record id, candidate answer, simulated human overall score 1-5)
labelled = [
    ("fe-1", "The applicant's stated gross annual income is GBP 72,000 per year.", 5),
    ("nr-1", "The LTV is 75% (300,000 / 400,000).", 5),
    (
        "ci-1",
        "No — if the firm cannot obtain the required information it must not make a personal recommendation.",
        5,
    ),
    (
        "au-1",
        "The document does not state a credit score, so it cannot be determined from this file.",
        5,
    ),
    ("nr-3", "The gross proceeds are GBP 8,740,000.", 2),  # forgot pence->pounds
    ("au-1", "The applicant's credit score is 742, which is considered good.", 1),  # fabricated
]

rows = []
for qid, answer, human in labelled:
    q = next(r for r in GOLDEN_DATASET if r["id"] == qid)
    r = evaluate_response(
        client,
        JUDGE_MODEL,
        question_id=q["id"],
        category=q["category"],
        document=DOCUMENTS[q["document"]],
        question=q["question"],
        expected=q["expected"],
        rubric=q["rubric"],
        actual=answer,
    )
    rows.append(
        {
            "id": qid,
            "answer": answer[:48] + "...",
            "human": human,
            "judge_overall": round(r.overall, 2),
        }
    )

df_corr = pd.DataFrame(rows)
corr = np.corrcoef(df_corr["human"], df_corr["judge_overall"])[0, 1]
print(df_corr.to_string(index=False))
print(f"\nPearson correlation (judge vs human): {corr:.3f}")

  id                                              answer  human  judge_overall
fe-1 The applicant's stated gross annual income is GB...      5            5.0
nr-1              The LTV is 75% (300,000 / 400,000)....      5            5.0
ci-1 No — if the firm cannot obtain the required info...      5            5.0
au-1 The document does not state a credit score, so i...      5            5.0
nr-3            The gross proceeds are GBP 8,740,000....      2            1.4
au-1 The applicant's credit score is 742, which is co...      1            1.1

Pearson correlation (judge vs human): 0.992


A correlation this high (the judge and the human agree closely, and crucially agree that the fabricated credit score is a 1) is what licenses us to use the judge in place of a human grader for the rest of the pipeline. In a real project you would label 30–50 examples for this step and re-check the correlation whenever you change the judge model or the rubric.

## 4. Running the eval suite

Now score the actual production system across the whole golden set. The "system under test" is a small financial-document Q&A answerer driven by the production prompt in `prompts.py`. `run_suite` answers every question and judges every answer, fanning the work across a small thread pool so 12 questions × 4 dimensions doesn't run serially.

In [10]:
from prompts import PROD_SYSTEM_PROMPT

print(PROD_SYSTEM_PROMPT)

You are a financial document analyst. Answer the user's question using ONLY the information in the provided document.

- Be precise and professional; this is a regulated setting.
- Show brief working for any calculation.
- If the document does not contain the information needed to answer, say clearly that it cannot be determined from the document. Never guess or fabricate a value.


In [11]:
def make_answerer(system_prompt: str, model: str = SUT_MODEL):
    """Build the system under test: answer a golden record from its source document."""

    def answer_fn(record: dict) -> str:
        response = client.messages.create(
            model=model,
            max_tokens=400,
            system=system_prompt,
            messages=[
                {
                    "role": "user",
                    "content": f"<document>\n{DOCUMENTS[record['document']]}\n</document>\n\n{record['question']}",
                }
            ],
        )
        return "".join(block.text for block in response.content if block.type == "text")

    return answer_fn


# This is the baseline: the production prompt scored over the full golden set.
baseline_results = run_suite(client, JUDGE_MODEL, make_answerer(PROD_SYSTEM_PROMPT), GOLDEN_DATASET)
print(f"Scored {len(baseline_results)} answers.")

Scored 12 answers.


The per-question results table is the raw material of the eval. Each row is one answer scored on every dimension:

In [12]:
df_results = pd.DataFrame([r.to_row() for r in baseline_results])
print(df_results.to_string(index=False))

question_id                  category  correctness  completeness  hallucination  tone  overall
       fe-1        factual_extraction            5             5              5     5     5.00
       fe-3        factual_extraction            5             5              5     5     5.00
       fe-5        factual_extraction            5             5              5     5     5.00
       nr-1       numerical_reasoning            5             5              5     5     5.00
       nr-3       numerical_reasoning            2             1              1     4     1.70
       nr-5       numerical_reasoning            5             5              5     5     5.00
       ci-1 compliance_interpretation            5             5              5     5     5.00
       ci-4 compliance_interpretation            5             5              4     5     4.75
       ci-5 compliance_interpretation            4             5              5     5     4.60
       au-1    ambiguous_unanswerable            5

Aggregate scores tell you where the system is strong and — more usefully — where it is weakest. Slicing by category is what turns "the model scores 4.6" into "the model is fine except on numerical reasoning", which is an actionable finding.

In [13]:
agg = aggregate(baseline_results)
print(f"Overall: {agg['overall']:.2f} / 5.00  (n={agg['n']})\n")

print("Per dimension:")
for dim, score in agg["per_dimension"].items():
    print(f"  {dim:<14} {score:.2f}")

print("\nPer category (overall):")
cat_scores = {cat: vals["overall"] for cat, vals in agg["per_category"].items()}
for cat, score in sorted(cat_scores.items(), key=lambda kv: kv[1]):
    print(f"  {cat:<26} {score:.2f}")

weakest = min(cat_scores, key=cat_scores.get)
print(f"\nWeakest category: {weakest} ({cat_scores[weakest]:.2f}) — where you'd focus next.")

Overall: 4.67 / 5.00  (n=12)

Per dimension:
  correctness    4.67
  completeness   4.67
  hallucination  4.58
  tone           4.92

Per category (overall):
  numerical_reasoning        3.90
  compliance_interpretation  4.78
  ambiguous_unanswerable     5.00
  factual_extraction         5.00

Weakest category: numerical_reasoning (3.90) — where you'd focus next.


This baseline is the reference every future change is measured against, so we persist it alongside the dataset. The CI script loads it to decide whether a candidate has regressed.

In [14]:
baseline_path = DATA_DIR / "baseline_results.json"
baseline_path.write_text(json.dumps([r.to_dict() for r in baseline_results], indent=2))
print(f"Saved baseline -> {baseline_path.name}")

Saved baseline -> baseline_results.json


## 5. Regression gates for CI/CD

A **regression gate** is a hard, automated threshold that decides whether a change is allowed to ship. It is the difference between "we have evals" and "our evals actually protect production": a score you look at is advisory; a gate that exits non-zero in CI is enforcement.

Good gate thresholds are **policy, set once by the team** — not knobs the model or the person shipping the change gets to negotiate. Ours encode three rules, each catching a different kind of regression:

- **Overall regression** — the weighted overall score may not drop more than 5% from baseline. Catches broad quality loss.
- **Hallucination floor** — the mean hallucination score must stay at or above 4.0/5.0. Grounding is non-negotiable in a regulated setting, so it gets an absolute floor rather than a relative one.
- **Category floor** — compliance answers must stay at or above 4.5/5.0. The category where an error is most costly gets the strictest floor.

In [15]:
print(json.dumps(DEFAULT_THRESHOLDS, indent=2))

{
  "max_overall_drop_pct": 5.0,
  "min_hallucination": 4.0,
  "category_floors": {
    "compliance_interpretation": 4.5
  }
}


### A change that should be blocked

Here is a plausible, well-intentioned prompt change: someone decides the assistant is "too hedgy" and rewrites the system prompt to always give a confident, definitive answer — dropping the instruction to decline when the document lacks the information. It will look *better* on the factual questions. Watch what it does to the unanswerable ones.

In [16]:
BAD_PROMPT = """\
You are a confident financial analyst. Always give the reader a clear, definitive, \
specific answer — clients dislike hedging. If a detail is not in the document, supply \
your best specific estimate rather than saying it is unavailable. Be concise and assured."""

bad_results = run_suite(client, JUDGE_MODEL, make_answerer(BAD_PROMPT), GOLDEN_DATASET)

bad_gate = check_regression_gate(baseline_results, bad_results)
print(bad_gate.summary())

[FAIL] regression gate — 2 of 3 checks breached
  [FAIL] overall regression: 4.67 -> 4.00 (-14.5%), max allowed drop 5.0%
  [FAIL] hallucination floor: 3.58 vs floor 4.00
  [ok  ] category floor: compliance_interpretation: 4.75 vs floor 4.50 (baseline 4.78)


The gate fails, and the per-rule report names exactly which thresholds broke and by how much. Removing the abstention guardrail makes the system fabricate answers to the unanswerable questions — here that trips both the overall-regression limit and the hallucination floor, while the compliance-category floor holds. That per-rule granularity is the point: the gate tells you *which* properties regressed (with the overall-drop rule as the robust backstop that catches broad quality loss even when no single floor does), so the fix is targeted rather than a blind revert.

Here is the damage at the question level — the unanswerable questions where the confident prompt invented answers instead of declining:


In [17]:
bad_by_id = {r.question_id: r for r in bad_results}
base_by_id = {r.question_id: r for r in baseline_results}
print(f"{'id':<6}{'category':<26}{'baseline_halluc':>16}{'bad_halluc':>12}")
for r in baseline_results:
    if r.category == "ambiguous_unanswerable":
        b = bad_by_id[r.question_id]
        print(
            f"{r.question_id:<6}{r.category:<26}{r.scores['hallucination']:>16}{b.scores['hallucination']:>12}"
        )

id    category                   baseline_halluc  bad_halluc
au-1  ambiguous_unanswerable                   5           2
au-3  ambiguous_unanswerable                   5           1
au-4  ambiguous_unanswerable                   5           1


### A change that should pass

Not every change is a regression. Here is a genuine improvement — the production prompt reworded to be crisper, but keeping the grounding guardrail intact. The gate should let it through. (We score it against the matching slice of the baseline; the same gate logic applies whether you run the full set or a subset.)

In [18]:
GOOD_PROMPT = """\
You are a financial document analyst working in a regulated setting. Answer strictly from \
the provided document, showing brief working for any calculation. State figures precisely. \
If the document does not contain what is needed to answer, say clearly that it cannot be \
determined from the document — do not guess or invent values."""

# One question per category is enough to show a passing gate.
good_slice_ids = ["fe-1", "nr-1", "ci-1", "au-1", "au-3", "au-4"]
good_slice = [r for r in GOLDEN_DATASET if r["id"] in good_slice_ids]
good_baseline = [r for r in baseline_results if r.question_id in good_slice_ids]

good_results = run_suite(client, JUDGE_MODEL, make_answerer(GOOD_PROMPT), good_slice)

good_gate = check_regression_gate(good_baseline, good_results)
print(good_gate.summary())

[PASS] regression gate — 0 of 3 checks breached
  [ok  ] overall regression: 5.00 -> 5.00 (-0.0%), max allowed drop 5.0%
  [ok  ] hallucination floor: 5.00 vs floor 4.00
  [ok  ] category floor: compliance_interpretation: 5.00 vs floor 4.50 (baseline 5.00)


## 6. Shadow deployment evaluation

A regression gate protects you against *your own* prompt changes. **Shadow deployment evaluation** answers a different question: is a candidate — a new prompt, or a new model version — good enough to *promote* to production?

The pattern: run the candidate on the same inputs as production — in the shadow, without its outputs reaching users — score both with the same judge, and compare. It is an A/B test where the "B" is scored by eval rather than served live. Here the candidate is an **improved prompt** that adds an explicit arithmetic-check step (the kind of change you would shadow-test before rolling out); a candidate *model version* would slot into exactly the same harness by changing the model argument instead of the prompt. The production side we already have — those scores are the baseline from §4 — so we only need to run the candidate.

In [19]:
# The candidate: production prompt plus an explicit "check the arithmetic" step.
CANDIDATE_PROMPT = PROD_SYSTEM_PROMPT + (
    "\n- For any calculation, work through it step by step and re-check the arithmetic "
    "and units before giving the final figure."
)

# A representative slice spanning all four categories. Production is already scored
# (it is the §4 baseline), so we reuse those results and only run the candidate.
shadow_ids = ["fe-3", "nr-3", "ci-4", "au-4"]
shadow_set = [r for r in GOLDEN_DATASET if r["id"] in shadow_ids]

prod_results = [r for r in baseline_results if r.question_id in shadow_ids]
cand_results = run_suite(
    client, JUDGE_MODEL, make_answerer(CANDIDATE_PROMPT, SUT_MODEL), shadow_set
)

prod_agg = aggregate(prod_results)
cand_agg = aggregate(cand_results)
delta = cand_agg["overall"] - prod_agg["overall"]

print(f"production (current prompt): {prod_agg['overall']:.2f}")
print(f"candidate  (arithmetic-check prompt): {cand_agg['overall']:.2f}")
print(f"delta: {delta:+.2f} ({100 * delta / prod_agg['overall']:+.1f}%)")

production (current prompt): 4.11
candidate  (arithmetic-check prompt): 4.88
delta: +0.76 (+18.5%)


The promotion decision is itself a policy, not a judgement call: **promote only when the candidate beats production by a meaningful margin AND does not breach any regression gate.** Requiring both stops you from promoting a candidate that scores higher overall but regressed on the one category you cannot afford to lose.

In [20]:
PROMOTE_MARGIN_PCT = 3.0

margin_pct = 100 * delta / prod_agg["overall"]
gate_vs_prod = check_regression_gate(prod_results, cand_results)

beats_margin = margin_pct >= PROMOTE_MARGIN_PCT
no_regression = gate_vs_prod.passed
promote = beats_margin and no_regression

print(f"beats prod by >= {PROMOTE_MARGIN_PCT}%?  {beats_margin}  ({margin_pct:+.1f}%)")
print(f"no regression gate breach? {no_regression}")
print(f"\nDecision: {'PROMOTE candidate to production' if promote else 'KEEP production'}")

beats prod by >= 3.0%?  True  (+18.5%)
no regression gate breach? True

Decision: PROMOTE candidate to production


Whichever way this particular comparison lands, the pattern is what matters: the promotion decision is reduced to two objective checks a script can make, run on the same judge and dataset as everything else. Swap the two prompts for two deployed model versions and this is exactly how you would gate a production rollout.

## 7. Wiring it into CI/CD

The pieces so far are functions in a notebook. To actually protect production they need to run automatically, on the changes that matter, and block the merge when the gate fails. That is what `run_eval.py` (shipped next to this notebook) is: a self-contained entry point that loads the golden dataset and baseline, runs the current production prompt through the judge, checks the gate, and **exits 0 on pass or 1 on fail** — the exit code CI reads to allow or block a merge.

Let's run it exactly as CI would. First, the clean production prompt on a quick stratified slice — this should pass and exit 0:

In [21]:
result = subprocess.run(  # noqa: S603
    [sys.executable, "run_eval.py", "--sample", "4"],
    capture_output=True,
    text=True,
    cwd=DATA_DIR,
)
print(result.stdout)
print(
    f"exit code: {result.returncode}  ({'PASS — merge allowed' if result.returncode == 0 else 'FAIL — merge blocked'})"
)

Running eval on 4 golden question(s) with model=claude-haiku-4-5...

overall score: 5.00/5.00  (n=4)
[PASS] regression gate — 0 of 3 checks breached
  [ok  ] overall regression: 5.00 -> 5.00 (-0.0%), max allowed drop 5.0%
  [ok  ] hallucination floor: 5.00 vs floor 4.00
  [ok  ] category floor: compliance_interpretation: 5.00 vs floor 4.50 (baseline 5.00)

exit code: 0  (PASS — merge allowed)


Now simulate the bad pull request from §5 — the "always be confident" prompt — by overriding the prompt via an environment variable, the way a CI job would run against the candidate under test. The script should exit **1** and block the merge:

In [22]:
bad_env = {
    **os.environ,
    "EVAL_SYSTEM_PROMPT": (
        "You are a confident financial analyst. Always give a clear, definitive, specific "
        "answer; never say information is unavailable — supply your best estimate instead."
    ),
}
result = subprocess.run(  # noqa: S603
    [sys.executable, "run_eval.py", "--sample", "4"],
    capture_output=True,
    text=True,
    cwd=DATA_DIR,
    env=bad_env,
)
print(result.stdout)
print(
    f"exit code: {result.returncode}  ({'PASS — merge allowed' if result.returncode == 0 else 'FAIL — merge blocked'})"
)

Running eval on 4 golden question(s) with model=claude-haiku-4-5...

overall score: 3.88/5.00  (n=4)
[FAIL] regression gate — 2 of 3 checks breached
  [FAIL] overall regression: 5.00 -> 3.88 (-22.5%), max allowed drop 5.0%
  [FAIL] hallucination floor: 3.50 vs floor 4.00
  [ok  ] category floor: compliance_interpretation: 5.00 vs floor 4.50 (baseline 5.00)

exit code: 1  (FAIL — merge blocked)


### The GitHub Actions workflow

The last piece is the trigger. You do **not** want to run the eval on every commit — it makes API calls and costs money. You want it on the pull requests that can actually change model behavior: the ones that touch your prompts or model configuration. GitHub Actions' `paths` filter expresses exactly that.

```yaml
# .github/workflows/eval-gate.yml
name: eval-gate
on:
  pull_request:
    paths:
      # Diff-based triggering: only run when something that affects model
      # behavior changes. A README edit does not pay for an eval run.
      - "evals/model_as_judge/prompts.py"
      - "evals/model_as_judge/**/*.py"

jobs:
  eval:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - run: pip install -U "anthropic>=0.109.0"
      - name: Run evaluation gate
        env:
          ANTHROPIC_API_KEY: ${{ secrets.ANTHROPIC_API_KEY }}
        # Exit 1 fails the job, which blocks the merge.
        run: python evals/model_as_judge/run_eval.py
```

Because the job fails when `run_eval.py` exits 1, a prompt change that breaches the gate cannot merge — the same failing run you just saw, now standing between a regression and production.

## Conclusion

We turned "it looks right" into a pipeline that answers "do you ship it?" with a number and a threshold:

| Piece | What it gives you |
| --- | --- |
| **Held-out golden dataset** (§2) | A reproducible measurement, with edge cases the happy path misses. |
| **Model-as-judge** (§3) | Scores for open-ended answers, decomposed into independent dimensions to avoid the halo effect. |
| **Eval suite** (§4) | Per-category scores that point at your weakest area. |
| **Regression gate** (§5) | Hard thresholds that block a bad change and name exactly which property regressed. |
| **Shadow evaluation** (§6) | An objective promote / keep decision for a candidate model. |
| **CI wiring** (§7) | An exit-0/exit-1 script that runs on prompt changes and blocks the merge. |

The idea that carries furthest is that **quality becomes a property you enforce, not one you hope for**. Once the gate is in CI, "did this change hurt anything?" stops being a question someone has to remember to ask — the pipeline asks it on every pull request, and answers with evidence.

### Where to take it next

- **Grow the golden set.** Twenty questions is a teaching size. Add examples every time a real failure reaches production — the bug you just fixed is the edge case your eval was missing.
- **Calibrate the judge periodically.** Re-run the human-correlation check (§3) whenever you change the judge model or a rubric; a judge that has drifted scores everything wrong.
- **Tune thresholds against real incidents.** Start where we did, then tighten or loosen based on what actually shipped and what actually broke.
- **Version the baseline.** When you deliberately accept a new quality level, commit a new `baseline_results.json`. The gate always measures against a known, reviewed reference.

### Related notebooks

- [Reproducing agentic search benchmark scores](https://github.com/anthropics/claude-cookbooks/blob/main/evals/agentic_search/reproduce_agentic_search_benchmarks.ipynb): the benchmark-reproduction companion to this production-eval cookbook.
- **The compliance-aware agent** (a companion cookbook in the Agent SDK series): builds the kind of regulated agent this pipeline would evaluate before every release.